In this assignment you will build and audit a Convolutional Neural Network (CNN) for image classification

Key Learning Goals:       
* Load and normalize the CIFAR-10 dataset
  
* Build a CNN from scratch (Conv2D + MaxPooling)

* Train and diagnose learning behavior via loss/accuracy curves

* Probe intermediate **feature maps** to interpret what the CNN learns

*  Audit **parameter efficiency** vs a dense MLP baseline

## Task 1: Image Data Understanding & Preprocessing (20 points)



In [ ]:
# ==================== Task 1: Dataset Preparation ====================

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import datasets

SEED = 42
np.random.seed(SEED)

# Load CIFAR-10 data
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixels (Divide by 255.0)
train_images = train_images.astype("float32") / _____   # e.g., 255.0
test_images  = test_images.astype("float32") / _____

# CIFAR-10 class names
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

print("Train images:", train_images.shape, "Train labels:", train_labels.shape)
print("Test images:", test_images.shape, "Test labels:", test_labels.shape)
print("Example label integer:", int(train_labels[0]))

# Quick sanity visualization
plt.figure(figsize=(6,6))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([]); plt.yticks([])
    plt.imshow(train_images[i])
    plt.title(class_names[int(train_labels[i])], fontsize=7)
plt.tight_layout()
plt.show()

Question:

*  What is the tensor shape of one image? What do the dimensions represent?

*  Why is normalization important for gradient-based training?

*  Why would a fully connected network be inefficient for raw images?

## Task 2: Building CNN (20 Points)

In [ ]:
## Student Code Required ##
# Instructions: Construct a Sequential model with at least two Conv2D layers.
# You must decide on the number of filters and
# the padding strategy ('same' vs. 'valid')

# ==================== Task 2: Custom CNN Architecture Construction ====================

from tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(SEED)

# Architecture choices
num_classes = _____.  # should be 10

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # choose filters, kernel size, padding
    layers.Conv2D(_____, (_____, _____), activation="relu", padding=_____),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(_____, (_____, _____), activation="relu", padding=_____),
    layers.MaxPooling2D((2,2)),

    # Optional: add a third conv block (leave blank if not used)
    # layers.Conv2D(_____, (_____, _____), activation="relu", padding=_____),
    # layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(_____, activation="relu"),
    layers.Dropout(_____),
    layers.Dense(num_classes, activation="softmax")
])

model.summary()

Question:

*   Which layers learn spatial features? Which layers perform classification?
*   What does pooling accomplish in terms of spatial information and robustness?

## Task 3: Intermediate Activation Visualization (30 Points)

In [ ]:
## Student Code Required ##
# Create a sub-model that outputs the feature maps of your
# first convolutional layer.
# This is how you 'audit' what the model is actually learning

## ==================== Task 3A: Compile + Train ====================

# Training hyperparameters
EPOCHS = _____ #(Recommended: 10)
BATCH  = _____ #(Recommended: 64)
LR     = _____ #(Recommended: 1e -3)

# Sparse labels setup
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_images, train_labels,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=1
)

In [ ]:
# ==================== Task 3B: Learning Curves ====================


import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(["Train", "Validation"])
plt.title("Loss Curves")
plt.show()

plt.figure()
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.title("Accuracy Curves")
plt.show()

In [ ]:
# ==================== Task 3C: Test Evaluation ====================

test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)
print("Test accuracy:", test_acc)

Question:

*  Do the curves show overfitting? Cite evidence from loss/accuracy curves.
*  If overfitting is present, name two changes you would try and why.

## Task 4: Feature Map Visualization (15 Points)

In [ ]:
# ==================== Task 4A: Feature Map Audit ====================

from tensorflow.keras import models

# Cchoose a Conv2D layer index to probe
# Tip: print model.layers to identify Conv2D layers.
LAYER_INDEX = _____

activation_model = models.Model(
    inputs=model.input,
    outputs=model.layers[LAYER_INDEX].output
)

sample = test_images[0:1]
activations = activation_model.predict(sample)   # shape: (1, H, W, C)

# Choose channels to visualize
ch1, ch2 = _____, _____

plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(activations[0,:,:,ch1]); plt.title(f"Channel {ch1}"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(activations[0,:,:,ch2]); plt.title(f"Channel {ch2}"); plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ==================== Helper: Grid Visualization ====================
# Use this to see ALL filters in a layer to find the "Cat Ear"
import math

def visualize_layer(layer_output, images_per_row=8):
    n_features = layer_output.shape[-1] # Number of features in the feature map
    size = layer_output.shape[1] # The feature map has shape (1, size, size, n_features)
    n_cols = n_features // images_per_row
    display_grid = np.zeros((size * n_cols, images_per_row * size))

    for col in range(n_cols):
        for row in range(images_per_row):
            channel_image = layer_output[0, :, :, col * images_per_row + row]
            # Post-process the feature to make it visually palatable
            channel_image -= channel_image.mean()
            channel_image /= channel_image.std()
            channel_image *= 64
            channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype('uint8')
            display_grid[col * size : (col + 1) * size,
                         row * size : (row + 1) * size] = channel_image

    scale = 1. / size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')
    plt.show()

# Visualize the first layer (Edges)
print("Layer 1 Feature Maps:")
visualize_layer(activations) # Assuming 'activations' is your output from the code above

Question: The "Cat Ear" Trace

**Don't just look for "blurry blobs." Look for Hierarchical Learning.**

In a CNN, early layers act as "Edge Detectors," while deeper layers act as "Object Detectors."

**Task: Feature Tracing**
1.  **Layer 1 (The Edge):** Look at the grid of filters from your first Convolutional layer. Identify one specific filter index (e.g., #7) that seems to activate strongly on **diagonal lines** or **textures**.
2.  **Layer 2/3 (The Object):** Change the `LAYER_INDEX` in the code above to point to your *last* Convolutional layer. Run the visualization again.
    * Can you find a filter that activates specifically on the **"ear" region** or the **"face" region** of the cat?

**Critical Thinking:**
If you *cannot* find a filter that clearly isolates the "cat ear" in the final layer, what does that tell you about the depth of your model?
* *Hint:* Is a 2-layer or 3-layer CNN deep enough to combine "edges" into "ears," or is it only seeing "textures"?


## Task 5: Parameter Efficiency Audit (15 points)

In [ ]:
# ==================== Task 5: Parameter Efficiency Audit ====================

# CNN parameters
cnn_params = model.count_params()
print("CNN parameters:", cnn_params)

# MLP estimate on raw 32x32x3 input
input_dim = 32 * 32 * 3

# Cchoose a hidden width for comparison
hidden = _____   # e.g., 100

mlp_params = (input_dim * hidden + hidden) + (hidden * 10 + 10)
print("MLP parameters (1 hidden layer):", mlp_params)
print("MLP / CNN parameter ratio:", mlp_params / cnn_params)

Question:

*   Why does a CNN use fewer parameters than an MLP on the same input?
*   Relate your parameter audit to convolution’s “weight sharing.”



## Optional for week 8

In [ ]:
# ==================== Optional Export for Week 8 Comparison ====================

import json

week7_summary = {
    "cnn_test_accuracy": float(test_acc),
    "cnn_params": int(cnn_params),
    "mlp_params": int(mlp_params)
}

with open("week7_cnn_baseline.json", "w") as f:
    json.dump(week7_summary, f, indent=2)

print("Saved week7_cnn_baseline.json for Week 8 comparison.")